# Final LSTM ASL Sign-to-Text Model Pipeline

## Project Highlights

This notebook implements the final LSTM-based Sign-to-Text recognition model using the Google ASL Signs landmark dataset.

The target pipeline is:

Google ASL Signs landmarks  
→ selected MediaPipe landmark features  
→ fixed 30-frame sequence  
→ LSTM temporal classifier  
→ 250 ASL sign classes  
→ Keras model  
→ TensorFlow Lite model

## Final Deliverables

The notebook generates the following final deliverables:

- `lstm_asl_model.keras`
- `lstm_asl_model.tflite`
- `asl_labels.json`
- `training_history.csv`
- `training_summary.json`

The main deployment output is:

`lstm_asl_model.tflite`

## Final Model Design

The model uses:

- 30 frames per sample
- 282 selected feature values per frame
- LSTM temporal modeling
- Dense Softmax classification head
- TensorFlow Lite conversion

## Engineering Enhancements

The final preprocessing and training pipeline includes:

- Selected landmark representation
- Missing value handling
- Coordinate normalization around the shoulder center
- Scale normalization using shoulder width
- Temporal resampling to 30 frames
- Temporal augmentation
- Landmark noise injection
- Random landmark masking
- Optional dominant-hand normalization

## Project Overview

 The notebook trains the LSTM model, saves the Keras version, converts it into TensorFlow Lite, tests the generated TFLite model, and packages the final deliverables for backend integration.

## Execution Environment

The model was developed and trained in Google Colab using a cloud-based Python environment with GPU acceleration.

Recommended runtime settings:

- Platform: Google Colab
- Runtime type: Python 3
- Hardware accelerator: GPU
- Recommended GPU: NVIDIA T4 or equivalent
- Deep learning framework: TensorFlow / Keras
- Dataset source: Google ASL Signs Kaggle dataset
- Final deployment format: TensorFlow Lite

The environment installs the required libraries directly inside the notebook, including TensorFlow, Kaggle API tools, pandas, NumPy, PyArrow, FastParquet, and scikit-learn.

For the final training run, keep:

`RUN_MODE = "FULL"`

## Cell 1 — Environment Setup

This cell installs the required Python packages and imports the main libraries used for training, evaluation, and TensorFlow Lite conversion.

In [ ]:
# Colab setup

!pip -q install kaggle pyarrow fastparquet scikit-learn pandas numpy tensorflow

import os
import sys
import json
import time
import shutil
import zipfile
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("Python version:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## Cell 2 — Full Final Model Configuration

This cell defines the final model settings, dataset paths, output files, sequence length, feature size, batch size, and augmentation settings for the full 250-class training run. The notebook is configured for the full final model training run.

In [ ]:
# Configuration

# RUN_MODE = "FULL"  # Final model training mode

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

COMPETITION_NAME = "asl-signs"

DATA_DIR = Path("/content/asl-signs")
OUTPUT_DIR = Path("/content/lstm_asl_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_KERAS_PATH = OUTPUT_DIR / "lstm_asl_model.keras"
MODEL_TFLITE_PATH = OUTPUT_DIR / "lstm_asl_model.tflite"
LABELS_JSON_PATH = OUTPUT_DIR / "asl_labels.json"
HISTORY_CSV_PATH = OUTPUT_DIR / "training_history.csv"
SUMMARY_JSON_PATH = OUTPUT_DIR / "training_summary.json"
ZIP_PATH = Path("/content/lstm_asl_training_outputs.zip")

SEQUENCE_LENGTH = 30
FEATURE_DIM = 282
NUM_CLASSES_TARGET = 250

# For full training.
# Set MAX_SAMPLES_PER_CLASS_FULL = None to use all available samples.
MAX_SAMPLES_PER_CLASS_FULL = 300
FULL_EPOCHS = 25

BATCH_SIZE = 64

USE_COORDINATE_NORMALIZATION = True
USE_SCALE_NORMALIZATION = True
USE_DOMINANT_HAND_NORMALIZATION = True

USE_AUGMENTATION = True
NOISE_STD = 0.01
LANDMARK_MASK_PROB = 0.05
TEMPORAL_CROP_MIN_RATIO = 0.75

print("Run mode:", RUN_MODE)
print("Quick test:", QUICK_TEST)

## Cell 3 — Kaggle Dataset Download or Detection

This cell checks whether the Google ASL Signs dataset already exists in Colab. If it is missing, it asks for `kaggle.json`, downloads the competition dataset, and extracts it.

In [ ]:
# Download or locate the Kaggle ASL Signs dataset

# Option A: Kaggle API download.
# Before running this cell:
# 1. Open Kaggle account settings.
# 2. Create API token.
# 3. Upload kaggle.json when prompted.
#
# You must have joined/accepted the competition rules on Kaggle first.

from google.colab import files

if not DATA_DIR.exists() or not (DATA_DIR / "train.csv").exists():
    print("Dataset not found at", DATA_DIR)
    print("Upload kaggle.json now, or manually place the dataset under /content/asl-signs.")

    uploaded = files.upload()

    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("kaggle.json was not uploaded. Upload kaggle.json or place the dataset manually.")

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    shutil.move("kaggle.json", kaggle_dir / "kaggle.json")
    os.chmod(kaggle_dir / "kaggle.json", 0o600)

    !kaggle competitions download -c asl-signs -p /content/asl_download

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    zip_files = list(Path("/content/asl_download").glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError("No Kaggle zip file was downloaded.")

    with zipfile.ZipFile(zip_files[0], "r") as z:
        z.extractall(DATA_DIR)

    print("Dataset extracted to:", DATA_DIR)
else:
    print("Dataset already found:", DATA_DIR)

print("Dataset files:")
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name)

## Cell 4 — Landmark Selection

This cell defines which MediaPipe Holistic landmarks are selected. The selected representation uses 94 landmarks, each with x, y, and z coordinates, producing 282 feature values per frame.

In [ ]:
# Landmark selection

# Kaggle ASL Signs landmark ordering:
# face:       0   to 467
# left_hand:  468 to 488
# pose:       489 to 521
# right_hand: 522 to 542

FACE_OFFSET = 0
LEFT_HAND_OFFSET = 468
POSE_OFFSET = 489
RIGHT_HAND_OFFSET = 522

TOTAL_LANDMARKS = 543
COORDS = 3

# 40 mouth / key face landmarks
MOUTH_FACE_INDICES = [
    0, 13, 14, 17, 37, 39, 40, 61, 78, 80,
    81, 82, 84, 87, 88, 91, 95, 146, 178, 181,
    185, 191, 267, 269, 270, 291, 308, 310, 311, 312,
    314, 317, 318, 321, 324, 375, 402, 405, 409, 415
]

# 12 upper-body / reference pose landmarks
UPPER_BODY_POSE_INDICES = [
    0,   # nose
    11, 12,  # shoulders
    13, 14,  # elbows
    15, 16,  # wrists
    23, 24,  # hips
    25, 26,  # knees
    27       # ankle / lower-body reference
]

LEFT_HAND_INDICES = list(range(21))
RIGHT_HAND_INDICES = list(range(21))

SELECTED_FLAT_INDICES = (
    [FACE_OFFSET + i for i in MOUTH_FACE_INDICES] +
    [LEFT_HAND_OFFSET + i for i in LEFT_HAND_INDICES] +
    [POSE_OFFSET + i for i in UPPER_BODY_POSE_INDICES] +
    [RIGHT_HAND_OFFSET + i for i in RIGHT_HAND_INDICES]
)

assert len(SELECTED_FLAT_INDICES) == 94, len(SELECTED_FLAT_INDICES)
assert len(SELECTED_FLAT_INDICES) * 3 == FEATURE_DIM

LEFT_SHOULDER = POSE_OFFSET + 11
RIGHT_SHOULDER = POSE_OFFSET + 12

LEFT_HAND_RANGE = np.arange(LEFT_HAND_OFFSET, LEFT_HAND_OFFSET + 21)
RIGHT_HAND_RANGE = np.arange(RIGHT_HAND_OFFSET, RIGHT_HAND_OFFSET + 21)

# Left/right body pairs used when mirroring dominant hand.
POSE_SWAP_PAIRS = [
    (POSE_OFFSET + 11, POSE_OFFSET + 12),
    (POSE_OFFSET + 13, POSE_OFFSET + 14),
    (POSE_OFFSET + 15, POSE_OFFSET + 16),
    (POSE_OFFSET + 23, POSE_OFFSET + 24),
    (POSE_OFFSET + 25, POSE_OFFSET + 26),
    (POSE_OFFSET + 27, POSE_OFFSET + 28),
]

print("Selected landmarks:", len(SELECTED_FLAT_INDICES))
print("Feature dimension per frame:", FEATURE_DIM)

## Cell 5 — Preprocessing and Augmentation Functions

This cell contains the core preprocessing logic: reading parquet landmark files, handling missing values, normalizing coordinates, normalizing scale, standardizing hand dominance, resampling to 30 frames, adding temporal augmentation, injecting landmark noise, and applying random landmark masking.

In [ ]:
# Preprocessing utilities

def read_landmark_parquet(parquet_path):
    """Read one Kaggle landmark parquet file and return array [frames, 543, 3]."""
    df = pd.read_parquet(parquet_path)

    # Expected columns: frame, type, landmark_index, x, y, z
    df = df[["frame", "type", "landmark_index", "x", "y", "z"]].copy()
    df[["x", "y", "z"]] = df[["x", "y", "z"]].fillna(0.0)

    frames = np.sort(df["frame"].unique())
    frame_to_index = {frame: idx for idx, frame in enumerate(frames)}
    df["frame_index"] = df["frame"].map(frame_to_index).astype(np.int32)

    offset_map = {
        "face": FACE_OFFSET,
        "left_hand": LEFT_HAND_OFFSET,
        "pose": POSE_OFFSET,
        "right_hand": RIGHT_HAND_OFFSET,
    }
    df["offset"] = df["type"].map(offset_map).astype(np.int32)
    df["flat_index"] = df["offset"] + df["landmark_index"].astype(np.int32)

    arr = np.zeros((len(frames), TOTAL_LANDMARKS, COORDS), dtype=np.float32)
    arr[
        df["frame_index"].to_numpy(),
        df["flat_index"].to_numpy(),
        :
    ] = df[["x", "y", "z"]].to_numpy(dtype=np.float32)

    return arr

def coordinate_and_scale_normalize(arr):
    """Normalize coordinates around shoulder center and optionally scale by shoulder width."""
    arr = arr.copy()
    left_shoulder = arr[:, LEFT_SHOULDER, :]
    right_shoulder = arr[:, RIGHT_SHOULDER, :]

    center = (left_shoulder + right_shoulder) / 2.0

    if USE_COORDINATE_NORMALIZATION:
        arr = arr - center[:, None, :]

    if USE_SCALE_NORMALIZATION:
        shoulder_width = np.linalg.norm(left_shoulder[:, :2] - right_shoulder[:, :2], axis=1)
        shoulder_width = np.where(shoulder_width < 1e-6, 1.0, shoulder_width)
        arr = arr / shoulder_width[:, None, None]

    return arr

def movement_energy(hand_arr):
    """Measure motion amount in a hand sequence."""
    if hand_arr.shape[0] < 2:
        return 0.0
    diff = hand_arr[1:] - hand_arr[:-1]
    return float(np.mean(np.linalg.norm(diff, axis=-1)))

def dominant_hand_normalize(arr):
    """
    If the left hand appears more dominant, mirror the sequence horizontally
    and swap left/right hand landmarks.
    """
    if not USE_DOMINANT_HAND_NORMALIZATION:
        return arr

    arr = arr.copy()
    left_energy = movement_energy(arr[:, LEFT_HAND_RANGE, :])
    right_energy = movement_energy(arr[:, RIGHT_HAND_RANGE, :])

    if left_energy > right_energy:
        # Mirror x-axis after body-centered normalization.
        arr[:, :, 0] *= -1.0

        # Swap left and right hands.
        left_copy = arr[:, LEFT_HAND_RANGE, :].copy()
        right_copy = arr[:, RIGHT_HAND_RANGE, :].copy()
        arr[:, LEFT_HAND_RANGE, :] = right_copy
        arr[:, RIGHT_HAND_RANGE, :] = left_copy

        # Swap selected left/right pose landmarks.
        for a, b in POSE_SWAP_PAIRS:
            temp = arr[:, a, :].copy()
            arr[:, a, :] = arr[:, b, :]
            arr[:, b, :] = temp

    return arr

def resample_sequence(seq, target_length=SEQUENCE_LENGTH):
    """Linearly resample sequence [frames, features] to fixed length."""
    n = seq.shape[0]
    if n == target_length:
        return seq.astype(np.float32)
    if n <= 1:
        return np.zeros((target_length, seq.shape[1]), dtype=np.float32)

    old_x = np.linspace(0.0, 1.0, n)
    new_x = np.linspace(0.0, 1.0, target_length)

    out = np.empty((target_length, seq.shape[1]), dtype=np.float32)
    for i in range(seq.shape[1]):
        out[:, i] = np.interp(new_x, old_x, seq[:, i])
    return out

def apply_temporal_augmentation(seq):
    """Random crop/resample to simulate faster or slower signing."""
    if not USE_AUGMENTATION:
        return seq

    n = seq.shape[0]
    if n <= 4:
        return seq

    ratio = random.uniform(TEMPORAL_CROP_MIN_RATIO, 1.0)
    crop_len = max(4, int(n * ratio))
    start = random.randint(0, max(0, n - crop_len))
    return seq[start:start + crop_len]

def apply_noise_and_masking(seq):
    """Apply noise and random landmark masking after resampling."""
    if not USE_AUGMENTATION:
        return seq

    seq = seq.copy()

    if NOISE_STD > 0:
        seq += np.random.normal(0, NOISE_STD, seq.shape).astype(np.float32)

    if LANDMARK_MASK_PROB > 0:
        seq_3d = seq.reshape(SEQUENCE_LENGTH, 94, 3)
        mask = np.random.rand(SEQUENCE_LENGTH, 94) < LANDMARK_MASK_PROB
        seq_3d[mask] = 0.0
        seq = seq_3d.reshape(SEQUENCE_LENGTH, FEATURE_DIM)

    return seq.astype(np.float32)

def parquet_to_features(parquet_path, augment=False):
    """
    Convert one parquet file into [30, 282] selected feature sequence.
    """
    arr = read_landmark_parquet(parquet_path)
    arr = coordinate_and_scale_normalize(arr)
    arr = dominant_hand_normalize(arr)

    selected = arr[:, SELECTED_FLAT_INDICES, :]
    seq = selected.reshape(selected.shape[0], FEATURE_DIM).astype(np.float32)
    seq = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0)

    if augment:
        seq = apply_temporal_augmentation(seq)

    seq = resample_sequence(seq, SEQUENCE_LENGTH)

    if augment:
        seq = apply_noise_and_masking(seq)

    return seq.astype(np.float32)

# Quick unit check. This will run after the dataset is loaded.
print("Preprocessing utilities ready.")

## Cell 6 — Metadata, Labels, and Train/Validation Split

This cell loads `train.csv` and the Kaggle label mapping, prepares file paths, filters available samples, creates the label list, saves `asl_labels.json`, and splits the data into training and validation sets.

In [ ]:
# Prepare metadata and label mapping

train_csv = DATA_DIR / "train.csv"
label_map_path = DATA_DIR / "sign_to_prediction_index_map.json"

if not train_csv.exists():
    raise FileNotFoundError(f"Missing {train_csv}")

if not label_map_path.exists():
    raise FileNotFoundError(f"Missing {label_map_path}")

df = pd.read_csv(train_csv)

with open(label_map_path, "r", encoding="utf-8") as f:
    sign_to_index = json.load(f)

# Ensure stable index order.
index_to_sign = {idx: sign for sign, idx in sign_to_index.items()}
labels_list = [index_to_sign[i] for i in range(len(index_to_sign))]

df["label"] = df["sign"].map(sign_to_index)
df["full_path"] = df["path"].apply(lambda p: str(DATA_DIR / p))

df = df[df["label"].notna()].copy()
df["label"] = df["label"].astype(int)

# Keep only rows whose files actually exist.
df = df[df["full_path"].apply(lambda p: Path(p).exists())].copy()

print("Total samples found:", len(df))
print("Total classes found:", df["sign"].nunique())

if MAX_SAMPLES_PER_CLASS_FULL is not None:
    df = (
        df.groupby("sign", group_keys=False)
          .apply(lambda x: x.sample(min(len(x), MAX_SAMPLES_PER_CLASS_FULL), random_state=SEED))
          .reset_index(drop=True)
    )

NUM_CLASSES = len(labels_list)
EPOCHS = FULL_EPOCHS

print("Training samples after filtering:", len(df))
print("Number of classes used:", NUM_CLASSES)
print("Epochs:", EPOCHS)

# Save labels JSON as a list, where index = class ID.
with open(LABELS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(labels_list, f, ensure_ascii=False, indent=2)

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    stratify=df["label"]
)

print("Train:", len(train_df), "Validation:", len(val_df))
print("Labels saved to:", LABELS_JSON_PATH)

## Cell 7 — Data Generator

This cell defines a Keras data generator that loads samples batch by batch. It converts parquet files into 30-frame by 282-feature sequences and returns one-hot encoded class labels.

In [ ]:
# Data generator

class ASLSequence(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size=BATCH_SIZE, shuffle=True, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.df) / self.batch_size)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, batch_idx):
        batch_indices = self.indices[batch_idx * self.batch_size:(batch_idx + 1) * self.batch_size]
        batch = self.df.iloc[batch_indices]

        x = np.zeros((len(batch), SEQUENCE_LENGTH, FEATURE_DIM), dtype=np.float32)
        y = np.zeros((len(batch),), dtype=np.int32)

        for i, (_, row) in enumerate(batch.iterrows()):
            x[i] = parquet_to_features(row["full_path"], augment=self.augment)
            y[i] = int(row["label"])

        return x, tf.keras.utils.to_categorical(y, num_classes=NUM_CLASSES)

train_gen = ASLSequence(train_df, augment=True, shuffle=True)
val_gen = ASLSequence(val_df, augment=False, shuffle=False)

# Test one batch.
x_batch, y_batch = train_gen[0]
print("Batch x:", x_batch.shape)
print("Batch y:", y_batch.shape)
print("Expected input:", (None, SEQUENCE_LENGTH, FEATURE_DIM))

## Cell 8 — LSTM Model Architecture

This cell builds the LSTM model. It uses feature projection layers, layer normalization, dropout, an LSTM temporal encoder, and a Dense Softmax output layer for sign classification.

In [ ]:
# Build LSTM model

def build_lstm_model(num_classes):
    inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM), name="landmark_sequence")

    x = tf.keras.layers.Masking(mask_value=0.0, name="masking")(inputs)

    # Per-frame feature projection. Dense layers operate on the last dimension
    # while preserving the temporal dimension.
    x = tf.keras.layers.Dense(512, name="feature_projection_1")(x)
    x = tf.keras.layers.LayerNormalization(name="layer_norm_1")(x)
    x = tf.keras.layers.Activation("relu", name="relu_1")(x)
    x = tf.keras.layers.Dropout(0.25, name="dropout_1")(x)

    x = tf.keras.layers.Dense(256, name="feature_projection_2")(x)
    x = tf.keras.layers.LayerNormalization(name="layer_norm_2")(x)
    x = tf.keras.layers.Activation("relu", name="relu_2")(x)
    x = tf.keras.layers.Dropout(0.25, name="dropout_2")(x)

    # LSTM temporal encoder.
    # unroll=True helps TensorFlow Lite conversion for a short fixed sequence length.
    x = tf.keras.layers.LSTM(
        256,
        return_sequences=False,
        dropout=0.20,
        recurrent_dropout=0.0,
        unroll=True,
        name="lstm_temporal_encoder"
    )(x)

    x = tf.keras.layers.Dense(256, activation="relu", name="classifier_dense")(x)
    x = tf.keras.layers.Dropout(0.30, name="classifier_dropout")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="sign_softmax")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="lstm_asl_sign_recognizer")
    return model

model = build_lstm_model(NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Cell 9 — Model Training

This cell trains the model using the prepared data generators. It saves the best Keras model, applies early stopping, reduces the learning rate when needed, and stores the training history.

In [ ]:
# Train

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_KERAS_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

start_time = time.time()

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print("Training time seconds:", training_time)

# Save final/best model.
model.save(MODEL_KERAS_PATH)
pd.DataFrame(history.history).to_csv(HISTORY_CSV_PATH, index=False)

print("Keras model saved to:", MODEL_KERAS_PATH)
print("History saved to:", HISTORY_CSV_PATH)

## Cell 10 — Final Model Evaluation

This cell evaluates the final trained model on the validation set and prints a sample prediction to confirm that the model is producing class probabilities correctly.

In [ ]:
# Evaluate

val_loss, val_acc = model.evaluate(val_gen, verbose=1)
print("Validation loss:", val_loss)
print("Validation accuracy:", val_acc)

# Optional prediction sample for report.
sample_x, sample_y = val_gen[0]
pred = model.predict(sample_x[:1])
pred_idx = int(np.argmax(pred[0]))
true_idx = int(np.argmax(sample_y[0]))

print("Predicted:", pred_idx, labels_list[pred_idx], "Confidence:", float(np.max(pred[0])))
print("True:", true_idx, labels_list[true_idx])

## Cell 11 — Final TensorFlow Lite Conversion

This cell converts the trained Keras model into TensorFlow Lite format. The main final deployment output is `lstm_asl_model.tflite`.

In [ ]:
# Convert to TensorFlow Lite

# Reload best saved model before conversion.
best_model = tf.keras.models.load_model(MODEL_KERAS_PATH)

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# The model uses an unrolled LSTM to improve compatibility with standard TFLite conversion.
tflite_model = converter.convert()

with open(MODEL_TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print("TFLite model saved to:", MODEL_TFLITE_PATH)
print("TFLite size MB:", MODEL_TFLITE_PATH.stat().st_size / (1024 * 1024))

## Cell 12 — Final TensorFlow Lite Inference Check

This cell loads the final `.tflite` model using `tf.lite.Interpreter`, checks the input and output shapes, runs one prediction, and maps the result back to the label list.

In [ ]:
# Test the TensorFlow Lite model

interpreter = tf.lite.Interpreter(model_path=str(MODEL_TFLITE_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite input details:", input_details)
print("TFLite output details:", output_details)

test_input = sample_x[:1].astype(np.float32)

expected_shape = input_details[0]["shape"]
print("Expected input shape:", expected_shape)
print("Test input shape:", test_input.shape)

interpreter.set_tensor(input_details[0]["index"], test_input)
interpreter.invoke()

tflite_output = interpreter.get_tensor(output_details[0]["index"])
tflite_pred_idx = int(np.argmax(tflite_output[0]))

print("TFLite predicted index:", tflite_pred_idx)
print("TFLite predicted sign:", labels_list[tflite_pred_idx])
print("TFLite confidence:", float(np.max(tflite_output[0])))

## Cell 13 — Save Final Summary and Package Deliverables

This cell saves the final training summary and packages the Keras model, TFLite model, labels, history, and summary into one ZIP file.

In [ ]:
# Save summary and package outputs

summary = {
    "run_mode": RUN_MODE,
    "dataset": "Google ASL Signs",
    "model_type": "LSTM",
    "sequence_length": SEQUENCE_LENGTH,
    "feature_dim_per_frame": FEATURE_DIM,
    "selected_landmarks": len(SELECTED_FLAT_INDICES),
    "num_classes": NUM_CLASSES,
    "train_samples": int(len(train_df)),
    "validation_samples": int(len(val_df)),
    "epochs_requested": int(EPOCHS),
    "validation_loss": float(val_loss),
    "validation_accuracy": float(val_acc),
    "keras_model": str(MODEL_KERAS_PATH),
    "tflite_model": str(MODEL_TFLITE_PATH),
    "tflite_size_mb": MODEL_TFLITE_PATH.stat().st_size / (1024 * 1024),
    "training_time_seconds": training_time,
    "augmentation": {
        "coordinate_normalization": USE_COORDINATE_NORMALIZATION,
        "scale_normalization": USE_SCALE_NORMALIZATION,
        "dominant_hand_normalization": USE_DOMINANT_HAND_NORMALIZATION,
        "temporal_augmentation": USE_AUGMENTATION,
        "noise_std": NOISE_STD,
        "landmark_mask_prob": LANDMARK_MASK_PROB,
    }
}

with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, indent=2))

# Create zip file.
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for file_path in [
        MODEL_KERAS_PATH,
        MODEL_TFLITE_PATH,
        LABELS_JSON_PATH,
        HISTORY_CSV_PATH,
        SUMMARY_JSON_PATH,
    ]:
        z.write(file_path, arcname=file_path.name)

print("Created ZIP:", ZIP_PATH)
print("ZIP size MB:", ZIP_PATH.stat().st_size / (1024 * 1024))

## Cell 14 — Download Final Model Deliverables

This cell downloads the final model deliverables ZIP file from Colab to your computer.

In [ ]:
# Download outputs from Colab

from google.colab import files

files.download(str(ZIP_PATH))

# Report Notes

You can describe the training pipeline as follows:

The final model was implemented as an LSTM-based temporal classifier for isolated ASL sign recognition. The system uses MediaPipe Holistic landmark sequences from the Google ASL Signs dataset. Each input sample is standardized into a fixed 30-frame temporal window. For every frame, a selected 282-dimensional landmark feature representation is constructed from the hands, upper body, and mouth-related facial landmarks.

The preprocessing pipeline applies missing-value handling, coordinate normalization around the shoulder center, scale normalization using shoulder width, temporal resampling, and selected augmentation strategies. The augmentation includes temporal variation, random landmark masking, landmark noise injection, and optional dominant-hand normalization.

The model architecture uses dense feature projection layers, layer normalization, dropout, and an LSTM temporal encoder. The final classification layer uses Softmax activation to classify the input into one of the supported ASL sign classes. After training, the Keras model is converted into TensorFlow Lite format to support lightweight real-time inference in the deployment environment.

The final deployment output is:

`lstm_asl_model.tflite`